In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import CHIP
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=20,adc_last_gap=10)
chip.clk_manager.set_cyc(delay1=20,delay2=100)
chip.add_compiler("../compiler/code/")

# 1.要写的权重

In [ ]:
dl = DataLoader()
data = dl.load_mat("../data/svm/multi_svm_params.mat")["Wb"].T
print(data.shape)
print(np.max(data),np.min(data))
plt.hist(data)
plt.show()
plot_cond(data[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")

cond_min,cond_max,cond_reference = 0,1100,550
cond_range = cond_max-cond_reference
weight_min,weight_max = -1,1
target_cond = data/weight_max*cond_range


dl = DataLoader()
data = dl.load_mat("../data/svm/multi_svm_params8395.mat")["Wb"].T
print(data.shape)
print(np.max(data),np.min(data))
plt.hist(data)
plt.show()
plot_cond(data[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")

cond_min,cond_max,cond_reference = 0,1100,550
cond_range = cond_max-cond_reference
weight_min,weight_max = -1,1
target_cond = data/weight_max*cond_range

In [ ]:
tg_map = np.array([1.5,1.55,1.6,1.65,1.7,1.75,1.8])
cond_map = np.array([350,390,450,490,550,590,650])
slope,intercept = np.polyfit(cond_map,tg_map, 1)

In [ ]:
svm_weight_pos = np.load("../data/svm/svm_weight_pos2.npz")
good_device = [np.array(svm_weight_pos["row"]),np.array(svm_weight_pos["col"])]

need_read = np.zeros((256,256),dtype=bool)
pos = np.ix_(good_device[0], good_device[1])
need_read[pos] = True


coordinates=np.load("../data/svm/coordinates.npy",)
for i in range(len(coordinates[0])):
    need_read[good_device[0][coordinates[1][i]],good_device[1][coordinates[0][i]]]=False

plot_cond(need_read,vmax=1)

# 2.写权重

In [ ]:
threshold = 60

set_v,reset_v = 1,1
reset_pulse_width,set_pulse_width = 10e-6,10e-6

need_read = np.zeros((256,256),dtype=bool)
need_read[pos] = True

coordinates=np.load("../data/svm/coordinates.npy",)
for i in range(len(coordinates[0])):
    if need_read[good_device[0][coordinates[1][i]],good_device[1][coordinates[0][i]]]:
        print(1)
        need_read[good_device[0][coordinates[1][i]],good_device[1][coordinates[0][i]]]=False

    
target = np.ones((256,256))
target[pos] = target_cond + cond_reference

tg_v = target*slope+intercept-0.2



for i in range(40):
    voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
    plot_cond((cond[pos]-cond_reference).T,vmin=-cond_range,vmax=cond_range)

    condition_reset = (cond > (target+threshold)) & need_read
    condition_set = (cond < (target-threshold)) & need_read
    print(i,"reset:",np.sum(condition_reset),"set",np.sum(condition_set))

    if i>0:
        set_v = set_v+0.025
        # if i<5:
        #     tg_v[condition_reset] -= 0.04
        #     tg_v[condition_set] += 0.04
        # elif i<10:
        #     tg_v[condition_reset] -= 0.02
        #     tg_v[condition_set] += 0.02
        # else:
        tg_v[condition_reset] -= 0.04
        tg_v[condition_set] += 0.02

    tg_v.clip(0,3,out=tg_v)

    chip.write_point2(crossbar=condition_reset,write_voltage=reset_v,tg=5,pulse_width=reset_pulse_width,set_device=False)
    chip.write_point2(crossbar=condition_reset,write_voltage=set_v,tg=tg_v,pulse_width=set_pulse_width,set_device=True)
    # # chip.ps.set_time_out(100)
    chip.write_point2(crossbar=condition_set,write_voltage=set_v,tg=tg_v,pulse_width=set_pulse_width,set_device=True)


# 3.测试读出来的权重推理效果

In [ ]:
cond=np.load("../data/svm/cond_sub_base_20250427_newpos.npy",)
cond = cond -cond_reference

W = cond/cond_range*weight_max
real11 = W
plot_cond(data[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")
plot_cond(W[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")
plot_cond(data[:128,:].reshape(32,20)-W[:128,:].reshape(32,20),vmin=weight_min,vmax=weight_max,label="value")

In [ ]:
need_read = np.zeros((256,256),dtype=bool)
need_read[pos] = True
voltage_base = chip.read_point2(crossbar=need_read,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)[pos]
np.save("../data/svm/cond_sub_base_20250427_newpos3.npy",cond)
cond = cond -cond_reference

W = cond/cond_range*weight_max
# W=np.zeros_like(W)
# coordinates=np.load("../data/svm/coordinates.npy",)
# for i in range(len(coordinates[0])):
#     W[coordinates[1][i],coordinates[0][i]]=1



plot_cond(data[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")
plot_cond(W[:128,:].reshape(32,20),vmin=-1,vmax=1,label="value")
plot_cond(data[:128,:].reshape(32,20)-W[:128,:].reshape(32,20),vmin=weight_min,vmax=weight_max,label="value")